# Grid MiniYOLO targets

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/07-targets/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
from miniyolo.targets import build_targets


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    scene = {"boxes": torch.tensor([[8., 12., 24., 28.], [40., 36., 56., 52.]]),
             "labels": torch.tensor([0, 1])}
    empty = {"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}
    target = build_targets([scene, empty], grid_size=4, image_size=64, num_classes=2)
    assert target["box"].shape == (2, 4, 4, 4)
    assert target["positive"].sum().item() == 2
    assert torch.allclose(target["box"][0, 1, 1], torch.tensor([0., .25, .25, .25]))
    assert torch.allclose(target["box"][0, 2, 3], torch.tensor([0., .75, .25, .25]))
    assert target["class_ids"][0, 1, 1].item() == 0
    assert target["class_ids"][0, 2, 3].item() == 1
    assert not target["positive"][1].any()
    collision = {"boxes": torch.tensor([[8., 12., 24., 28.], [10., 14., 26., 30.]]),
                 "labels": torch.tensor([0, 0])}
    try:
        build_targets([collision], grid_size=4, image_size=64, num_classes=2)
    except ValueError:
        print("same-cell collision rejected")
    else:
        raise AssertionError("Two objects silently overwrote one slot")
    print("positive indices (b,y,x)", target["positive"].nonzero().tolist())
    print("red target", target["box"][0, 1, 1].tolist())
    print("blue target", target["box"][0, 2, 3].tolist())
    print("positive/negative counts", 2, int((~target["positive"]).sum()))


if __name__ == '__main__':
    main()


same-cell collision rejected
positive indices (b,y,x) [[0, 1, 1], [0, 2, 3]]
red target [0.0, 0.25, 0.25, 0.25]
blue target [0.0, 0.75, 0.25, 0.25]
positive/negative counts 2 30
